In [1]:
import json
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from lime.lime_text import LimeTextExplainer
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

# Function to determine label
def get_label(source_url, state):
    state_lower = state.lower()
    if any(keyword in source_url.lower() for keyword in ['elections', 'sos', 'vote']) and state_lower in source_url.lower():
        # primary
        return 0
    else:
        # secondary
        return 1

# Load data
data_dir = 'election-dataset-us-main/election-dataset-us-main/data-us'
all_data = []
for file in os.listdir(data_dir):
    if file.endswith('_qa.json'):
        with open(os.path.join(data_dir, file), 'r') as f:
            data = json.load(f)
            if 'state' in data and 'questions' in data:
                state = data['state'].upper()
                for q in data['questions']:
                    text = q['q'] + ' ' + q['a']
                    label = get_label(q['s'], state)
                    all_data.append({'State': state, 'text': text, 'label': label})

df = pd.DataFrame(all_data)
df.to_csv('us-allstates-voter-faqs.csv', index=False)

In [2]:
# Vectorize
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['text'])
y = df['label']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [3]:
# Logistic regression model
model1 = LogisticRegression(random_state=42)
model1.fit(X_train, y_train)
pred1 = model1.predict(X_test)
prob1 = model1.predict_proba(X_test)[:, 1]
f1_1 = f1_score(y_test, pred1)
auc_1 = roc_auc_score(y_test, prob1)
print(f"Logistic Regression F1: {f1_1:.3f}, roc auc: {auc_1:.3f}")

Logistic Regression F1: 0.427, roc auc: 0.812


In [4]:
# Random forest model
model2 = RandomForestClassifier(random_state=42)
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)
prob2 = model2.predict_proba(X_test)[:, 1]
f1_2 = f1_score(y_test, pred2)
auc_2 = roc_auc_score(y_test, prob2)
print(f"Random forest F1: {f1_2:.3f}, roc auc: {auc_2:.3f}")

Random forest F1: 0.438, roc auc: 0.808


In [5]:
# Best model
if f1_1 >= f1_2:
    best_model = model1
    best_name = 'LogisticRegression'
else:
    best_model = model2
    best_name = 'RandomForest'

print(f"Best model: {best_name}")

Best model: RandomForest


In [6]:
# Explainer from lime
explainer = LimeTextExplainer(class_names=['primary', 'secondary'])

# Sample data
states = df['State'].unique()
sample_data = []
for state in states:
    primary_df = df[(df['State'] == state) & (df['label'] == 0)]
    secondary_df = df[(df['State'] == state) & (df['label'] == 1)]
    if not primary_df.empty:
        sample_data.append(primary_df.sample(1).iloc[0])
    if not secondary_df.empty:
        sample_data.append(secondary_df.sample(1).iloc[0])

# Fill to 100
while len(sample_data) < 100:
    sample_data.append(df.sample(1).iloc[0])

sample_df = pd.DataFrame(sample_data)

In [7]:
# Results
results = []
for idx, row in sample_df.iterrows():
    text = row['text']
    true_class = row['label']
    pred_class = best_model.predict(vectorizer.transform([text]))[0]
    correct = 'Yes' if pred_class == true_class else 'No'
    exp = explainer.explain_instance(text, lambda x: best_model.predict_proba(vectorizer.transform(x)), num_features=3)
    top_features = exp.as_list()
    features_str = '; '.join([f"{word}: {weight:.3f}" for word, weight in top_features])
    results.append({
        'Text (Q/A)': text,
        'state': row['State'],
        'predicted class': pred_class,
        'true class': true_class,
        'Top three words and their weights': features_str
    })
print(results)

results_df = pd.DataFrame(results)
results_df.to_csv(f'us-allstates-voter-faqs-classifier-{best_name}-sampleoutput.csv', index=False)

[{'Text (Q/A)': 'I have received multiple absentee ballot applications, do I have to complete them all? No, one application is all that is required.', 'state': 'AK', 'predicted class': np.int64(1), 'true class': 1, 'Top three words and their weights': 'have: 0.190; is: 0.156; No: 0.122'}, {'Text (Q/A)': 'When are the upcoming election dates in Alabama? Your next election date can be found here https://www.vote411.org/alabamaYou can find more information about upcoming elections in Alabama HERE.', 'state': 'AL', 'predicted class': np.int64(0), 'true class': 0, 'Top three words and their weights': 'Alabama: -0.169; upcoming: -0.073; found: 0.052'}, {'Text (Q/A)': 'Where are ballot drop boxes located? No drop boxes available. You can hand-deliver your ballot to the office of the county clerk.', 'state': 'AR', 'predicted class': np.int64(0), 'true class': 0, 'Top three words and their weights': 'clerk: -0.055; located: 0.037; county: -0.031'}, {'Text (Q/A)': 'Where is my polling location? 

In [8]:
# Make it into a pdf
# Also formats all output text so it doesn't go off the pdf page; this formatting code was made by ai
from reportlab.lib.utils import simpleSplit

c = canvas.Canvas(f'us-allstates-voter-faqs-classifier-{best_name}-sampleoutput-dump.pdf', pagesize=letter)

def draw_wrapped_text(c, x, y, text, max_width, line_height=14, font_name='Helvetica', font_size=10):
    c.setFont(font_name, font_size)
    lines = simpleSplit(text, font_name, font_size, max_width)
    for line in lines:
        if y < 50:
            c.showPage()
            c.setFont(font_name, font_size)
            y = 750
        c.drawString(x, y, line)
        y -= line_height
    return y

y_position = 750
max_width = 500
amount_of_lines = 0
for idx, row in sample_df.iterrows():
    amount_of_lines = amount_of_lines + 1
    text = row['text']
    exp = explainer.explain_instance(text, lambda x: best_model.predict_proba(vectorizer.transform(x)), num_features=10)
    explanation_text = str(exp.as_list())
    
    y_position = draw_wrapped_text(c, 50, y_position, f"Text: {text}", max_width)
    y_position -= 4
    y_position = draw_wrapped_text(c, 50, y_position, f"Explanation: {explanation_text}", max_width)
    y_position -= 20
    if y_position < 50:
        c.showPage()
        y_position = 750

c.save()
# Number of rows stays at 100 as per the rubrik, just edited to all be readable
print(f"Amount of lines: {amount_of_lines}")
print("Done")

Amount of lines: 100
Done
